# W4 Day3 (04/30 周三): 数据库 + 消息队列基础

## 学习目标
- **理论 (1h)**: SQL 高频 (JOIN/窗口函数/索引); Redis 数据结构+缓存策略; Kafka 核心概念
- **实践 (2h)**: SQL 练习 + Redis 实操 + Kafka 概念理解
- **产出物**: SQL+Redis+Kafka 笔记

## 核心问题 (面试高频)
1. SQL JOIN 有几种？INNER JOIN 和 LEFT JOIN 的区别？
2. 窗口函数和 GROUP BY 的区别？ROW_NUMBER vs RANK？
3. 什么时候该建索引？B-tree 索引的原理？
4. Redis 和 Memcached 的区别？Redis 支持哪些数据结构？
5. 缓存穿透、缓存击穿、缓存雪崩分别是什么？怎么解决？
6. Kafka 的 Topic/Partition/Consumer Group 分别是什么？
7. Kafka 怎么保证消息不丢？Exactly-once 怎么实现？

---

## 目录
1. [SQL JOIN 类型](#1)
2. [窗口函数](#2)
3. [索引](#3)
4. [LeetCode SQL 精选](#4)
5. [Redis 基础](#5)
6. [Redis 缓存策略](#6)
7. [Kafka 核心概念](#7)
8. [综合: 事件驱动管线](#8)
9. [总结 & 思考题](#9)

In [ ]:
import sqlite3
import time
import json
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict, deque

TERRACOTTA = '#c2553a'
SAGE = '#5a6b4a'
INK = '#2d2a26'

# Redis
try:
    import redis
    HAS_REDIS = True
except ImportError:
    HAS_REDIS = False

print(f"sqlite3: available (Python built-in)")
print(f"redis-py: {'available' if HAS_REDIS else 'not installed (will simulate)'}")
print("=" * 60)

---
## 1. SQL JOIN 类型 <a id='1'></a>

```
表A (employees)          表B (departments)
| id | name  | dept_id |  | id | dept_name   |
|----|-------|---------|  |----|-------------|
| 1  | Alice | 10      |  | 10 | Engineering |
| 2  | Bob   | 20      |  | 20 | Marketing   |
| 3  | Carol | 10      |  | 30 | Sales       |
| 4  | Dave  | NULL    |  |    |             |
```

| JOIN 类型 | 说明 | 结果 |
|---|---|---|
| **INNER JOIN** | 只保留两边都匹配的行 | Alice/Carol(10), Bob(20) |
| **LEFT JOIN** | 保留左表所有行，右表不匹配填 NULL | + Dave(NULL) |
| **RIGHT JOIN** | 保留右表所有行 (SQLite 不支持) | + Sales(无员工) |
| **FULL OUTER JOIN** | 保留两边所有行 | Dave + Sales |
| **CROSS JOIN** | 笛卡尔积 (A×B 所有组合) | 4×3 = 12 行 |

In [ ]:
# ============ SQL JOIN 演示 ============

conn = sqlite3.connect(':memory:')
cur = conn.cursor()

# 创建表
cur.execute('''CREATE TABLE employees (
    id INTEGER PRIMARY KEY, name TEXT, dept_id INTEGER
)''')
cur.execute('''CREATE TABLE departments (
    id INTEGER PRIMARY KEY, dept_name TEXT
)''')

cur.executemany('INSERT INTO employees VALUES (?,?,?)', [
    (1, 'Alice', 10), (2, 'Bob', 20), (3, 'Carol', 10), (4, 'Dave', None)
])
cur.executemany('INSERT INTO departments VALUES (?,?)', [
    (10, 'Engineering'), (20, 'Marketing'), (30, 'Sales')
])
conn.commit()

# INNER JOIN
print("INNER JOIN: 只保留两边都匹配的行")
for row in cur.execute('''
    SELECT e.name, d.dept_name FROM employees e
    INNER JOIN departments d ON e.dept_id = d.id
'''):
    print(f"  {row[0]:8s} → {row[1]}")

# LEFT JOIN
print("\nLEFT JOIN: 保留左表所有行")
for row in cur.execute('''
    SELECT e.name, COALESCE(d.dept_name, '无部门') FROM employees e
    LEFT JOIN departments d ON e.dept_id = d.id
'''):
    print(f"  {row[0]:8s} → {row[1]}")

# 找没有员工的部门 (LEFT JOIN + IS NULL)
print("\n没有员工的部门 (反连接):")
for row in cur.execute('''
    SELECT d.dept_name FROM departments d
    LEFT JOIN employees e ON d.id = e.dept_id
    WHERE e.id IS NULL
'''):
    print(f"  {row[0]}")

# 每个部门的人数 (GROUP BY)
print("\n每个部门的人数 (GROUP BY):")
for row in cur.execute('''
    SELECT d.dept_name, COUNT(e.id) as cnt FROM departments d
    LEFT JOIN employees e ON d.id = e.dept_id
    GROUP BY d.id ORDER BY cnt DESC
'''):
    print(f"  {row[0]:12s}: {row[1]} 人")

conn.close()

---
## 2. 窗口函数 <a id='2'></a>

### 窗口函数 vs GROUP BY

| | GROUP BY | 窗口函数 |
|---|---|---|
| **结果行数** | 合并为组 (减少行数) | 保持原始行数 |
| **可访问** | 只能访问聚合结果 | 可同时访问原始行和聚合值 |
| **语法** | `GROUP BY col` | `OVER (PARTITION BY col)` |

### 常用窗口函数

| 函数 | 用途 |
|---|---|
| `ROW_NUMBER()` | 行号 (不重复) |
| `RANK()` | 排名 (有并列会跳号: 1,2,2,4) |
| `DENSE_RANK()` | 排名 (有并列不跳号: 1,2,2,3) |
| `LAG(col, n)` | 前 n 行的值 |
| `LEAD(col, n)` | 后 n 行的值 |
| `SUM() OVER()` | 累计/分组求和 |
| `NTILE(n)` | 分成 n 组 |

In [ ]:
# ============ 窗口函数示例 ============

conn = sqlite3.connect(':memory:')
cur = conn.cursor()

cur.execute('''CREATE TABLE sales (
    id INTEGER PRIMARY KEY, employee TEXT, department TEXT,
    month TEXT, amount REAL
)''')

sales_data = [
    (1, 'Alice', 'Eng', 'Jan', 1000), (2, 'Alice', 'Eng', 'Feb', 1200),
    (3, 'Bob', 'Mkt', 'Jan', 800), (4, 'Bob', 'Mkt', 'Feb', 950),
    (5, 'Carol', 'Eng', 'Jan', 1100), (6, 'Carol', 'Eng', 'Feb', 1300),
    (7, 'Dave', 'Mkt', 'Jan', 700), (8, 'Dave', 'Mkt', 'Feb', 850),
    (9, 'Eve', 'Eng', 'Jan', 900), (10, 'Eve', 'Eng', 'Feb', 1050),
]
cur.executemany('INSERT INTO sales VALUES (?,?,?,?,?)', sales_data)
conn.commit()

# ROW_NUMBER: 每个部门按销售额排名
print("ROW_NUMBER: 每个部门内按销售额排名")
for row in cur.execute('''
    SELECT employee, department, month, amount,
           ROW_NUMBER() OVER (PARTITION BY department ORDER BY amount DESC) as rn
    FROM sales WHERE month = 'Jan'
'''):
    print(f"  {row[0]:8s} {row[1]:4s} {row[3]:>7.0f} → 第{row[4]}名")

# RANK vs DENSE_RANK
print("\nRANK vs DENSE_RANK (演示并列):")
cur.execute('DELETE FROM sales')
cur.executemany('INSERT INTO sales VALUES (?,?,?,?,?)', [
    (1,'A','X','Jan',100),(2,'B','X','Jan',90),(3,'C','X','Jan',90),
    (4,'D','X','Jan',80),(5,'E','X','Jan',70)
])
for row in cur.execute('''
    SELECT employee, amount,
           RANK() OVER (ORDER BY amount DESC) as r,
           DENSE_RANK() OVER (ORDER BY amount DESC) as dr
    FROM sales
'''):
    print(f"  {row[0]}: amount={row[1]:>3.0f}  RANK={row[2]}  DENSE_RANK={row[3]}")

# LAG: 环比增长
print("\nLAG: 月度环比增长")
conn.close()
conn = sqlite3.connect(':memory:')
cur = conn.cursor()
cur.execute('CREATE TABLE revenue (month TEXT, amount REAL)')
cur.executemany('INSERT INTO revenue VALUES (?,?)', [
    ('Jan',1000),('Feb',1200),('Mar',1100),('Apr',1500),('May',1400)
])
for row in cur.execute('''
    SELECT month, amount,
           LAG(amount, 1) OVER (ORDER BY rowid) as prev,
           ROUND((amount - LAG(amount, 1) OVER (ORDER BY rowid)) * 100.0 /
                 LAG(amount, 1) OVER (ORDER BY rowid), 1) as growth_pct
    FROM revenue
'''):
    prev = f"{row[2]:.0f}" if row[2] else "N/A"
    growth = f"{row[3]:+.1f}%" if row[3] else "N/A"
    print(f"  {row[0]}: {row[1]:>5.0f}  prev={prev:>5s}  growth={growth:>7s}")

conn.close()

---
## 3. 索引 <a id='3'></a>

### B-tree 索引原理

```
          [50]              ← 根节点
         /    \
     [20,35]  [65,80]       ← 内部节点
    /  |  \   /  |  \
  [10][25][40][55][70][90]  ← 叶子节点 (有序链表)
```

### 什么时候建索引？

| 场景 | 建索引 | 不建索引 |
|---|---|---|
| WHERE 频繁查询的列 | ✓ | |
| JOIN 的连接键 | ✓ | |
| ORDER BY 的列 | ✓ | |
| 高选择性 (不同值多) | ✓ | |
| 小表 (< 1000 行) | | ✓ |
| 频繁 INSERT/UPDATE | | ✓ (索引维护开销) |
| 低选择性 (如性别) | | ✓ |

In [ ]:
# ============ 索引性能实验 ============

conn = sqlite3.connect(':memory:')
cur = conn.cursor()

# 创建大表
cur.execute('CREATE TABLE users (id INTEGER, name TEXT, email TEXT, city TEXT, age INTEGER)')
np.random.seed(42)
cities = ['Beijing', 'Shanghai', 'Shenzhen', 'Guangzhou', 'Hangzhou']
data = [(i, f'user_{i}', f'user_{i}@test.com', np.random.choice(cities), np.random.randint(18, 65))
        for i in range(100000)]
cur.executemany('INSERT INTO users VALUES (?,?,?,?,?)', data)
conn.commit()

# 无索引查询
t0 = time.time()
for _ in range(100):
    cur.execute('SELECT * FROM users WHERE city = ?', ('Beijing',)).fetchall()
time_no_idx = time.time() - t0

# 创建索引
cur.execute('CREATE INDEX idx_city ON users(city)')
conn.commit()

# 有索引查询
t0 = time.time()
for _ in range(100):
    cur.execute('SELECT * FROM users WHERE city = ?', ('Beijing',)).fetchall()
time_idx = time.time() - t0

print(f"索引性能对比 (10万行, 100次查询):")
print(f"  无索引: {time_no_idx:.4f}s")
print(f"  有索引: {time_idx:.4f}s")
print(f"  加速比: {time_no_idx / max(time_idx, 0.0001):.1f}x")

# EXPLAIN QUERY PLAN
print("\nEXPLAIN QUERY PLAN:")
print("无索引:")
for row in cur.execute('EXPLAIN QUERY PLAN SELECT * FROM users WHERE city = ?', ('Beijing',)):
    print(f"  {row}")

cur.execute('DROP INDEX idx_city')
print("\n有索引 (idx_city):")
cur.execute('CREATE INDEX idx_city ON users(city)')
for row in cur.execute('EXPLAIN QUERY PLAN SELECT * FROM users WHERE city = ?', ('Beijing',)):
    print(f"  {row}")

conn.close()

---
## 4. LeetCode SQL 精选 <a id='4'></a>

| # | 题目 | 考点 |
|---|---|---|
| 175 | Combine Two Tables | LEFT JOIN |
| 176 | Second Highest Salary | 子查询/LIMIT |
| 178 | Rank Scores | DENSE_RANK |
| 180 | Consecutive Numbers | LAG/LEAD |
| 181 | Employees Earning More | Self JOIN |
| 182 | Duplicate Emails | GROUP BY + HAVING |
| 183 | Customers Who Never Order | LEFT JOIN + IS NULL |
| 184 | Department Highest Salary | 窗口函数 + 子查询 |
| 196 | Delete Duplicate Emails | DELETE + 子查询 |
| 197 | Rising Temperature | LAG/DATEDIFF |

In [ ]:
# ============ LeetCode SQL 解答 ============

conn = sqlite3.connect(':memory:')
cur = conn.cursor()

# 175: Combine Two Tables
cur.execute('CREATE TABLE Person (id INT, firstName TEXT, lastName TEXT)')
cur.execute('CREATE TABLE Address (personId INT, city TEXT, state TEXT)')
cur.executemany('INSERT INTO Person VALUES (?,?,?)', [(1,'Wang','Allen'),(2,'Alice','Bob')])
cur.executemany('INSERT INTO Address VALUES (?,?,?)', [(1,'New York','NY'),(3,'Beijing','BJ')])
conn.commit()

print("#175 Combine Two Tables (LEFT JOIN):")
for row in cur.execute('''
    SELECT p.firstName, p.lastName, COALESCE(a.city, 'N/A'), COALESCE(a.state, 'N/A')
    FROM Person p LEFT JOIN Address a ON p.id = a.personId
'''):
    print(f"  {row[0]} {row[1]} → {row[2]}, {row[3]}")

# 176: Second Highest Salary
cur.execute('CREATE TABLE Employee (id INT, salary INT)')
cur.executemany('INSERT INTO Employee VALUES (?,?)', [(1,100),(2,200),(3,300)])
conn.commit()

print("\n#176 Second Highest Salary:")
for row in cur.execute('''
    SELECT DISTINCT salary FROM Employee ORDER BY salary DESC LIMIT 1 OFFSET 1
'''):
    print(f"  {row[0]}")

# 178: Rank Scores
cur.execute('CREATE TABLE Scores (id INT, score REAL)')
cur.executemany('INSERT INTO Scores VALUES (?,?)', [(1,3.50),(2,3.65),(3,4.00),(4,3.85),(5,4.00),(6,3.65)])
conn.commit()

print("\n#178 Rank Scores (DENSE_RANK):")
for row in cur.execute('''
    SELECT score, DENSE_RANK() OVER (ORDER BY score DESC) as rank FROM Scores
'''):
    print(f"  {row[0]:.2f} → Rank {row[1]}")

# 182: Duplicate Emails
cur.execute('CREATE TABLE Email (id INT, email TEXT)')
cur.executemany('INSERT INTO Email VALUES (?,?)', [(1,'a@b.com'),(2,'c@d.com'),(3,'a@b.com')])
conn.commit()

print("\n#182 Duplicate Emails (GROUP BY + HAVING):")
for row in cur.execute('SELECT email FROM Email GROUP BY email HAVING COUNT(*) > 1'):
    print(f"  {row[0]}")

# 183: Customers Who Never Order
cur.execute('CREATE TABLE Customers (id INT, name TEXT)')
cur.execute('CREATE TABLE Orders (id INT, customerId INT)')
cur.executemany('INSERT INTO Customers VALUES (?,?)', [(1,'Joe'),(2,'Henry'),(3,'Sam'),(4,'Max')])
cur.executemany('INSERT INTO Orders VALUES (?,?)', [(1,3),(2,1)])
conn.commit()

print("\n#183 Customers Who Never Order:")
for row in cur.execute('''
    SELECT name FROM Customers c LEFT JOIN Orders o ON c.id = o.customerId
    WHERE o.id IS NULL
'''):
    print(f"  {row[0]}")

conn.close()

---
## 5. Redis 基础 <a id='5'></a>

### Redis vs Memcached

| | Redis | Memcached |
|---|---|---|
| **数据结构** | String/List/Set/Hash/SortedSet/Stream | 只有 String |
| **持久化** | RDB + AOF | 无 |
| **集群** | 原生 Cluster | 客户端分片 |
| **发布订阅** | 支持 | 不支持 |
| **Lua 脚本** | 支持 | 不支持 |
| **内存效率** | 较低 (结构开销) | 较高 |

### Redis 数据结构

| 结构 | 底层实现 | 典型用途 |
|---|---|---|
| **String** | SDS (Simple Dynamic String) | 缓存、计数器、分布式锁 |
| **List** | 双向链表 / 压缩列表 | 消息队列、最新动态 |
| **Set** | 哈希表 / 整数集合 | 标签、共同好友 |
| **Hash** | 哈希表 / 压缩列表 | 对象属性 |
| **Sorted Set** | 跳表 + 哈希表 | 排行榜、延迟队列 |
| **Stream** | Radix Tree | 消息队列 (类似 Kafka) |

In [ ]:
# ============ Redis 数据结构演示 (模拟) ============

class FakeRedis:
    """Redis 模拟器 (无需真实 Redis 服务)"""
    def __init__(self):
        self.data = {}
        self.types = {}
    
    def set(self, key, value, ex=None):
        self.data[key] = value
        self.types[key] = 'string'
        return True
    
    def get(self, key):
        return self.data.get(key)
    
    def incr(self, key):
        self.data[key] = int(self.data.get(key, 0)) + 1
        self.types[key] = 'string'
        return self.data[key]
    
    def lpush(self, key, *values):
        if key not in self.data:
            self.data[key] = deque()
        for v in values:
            self.data[key].appendleft(v)
        self.types[key] = 'list'
        return len(self.data[key])
    
    def lrange(self, key, start, end):
        lst = list(self.data.get(key, []))
        if end == -1:
            end = len(lst)
        return lst[start:end+1]
    
    def hset(self, key, mapping):
        if key not in self.data:
            self.data[key] = {}
        self.data[key].update(mapping)
        self.types[key] = 'hash'
        return len(mapping)
    
    def hgetall(self, key):
        return self.data.get(key, {})
    
    def sadd(self, key, *values):
        if key not in self.data:
            self.data[key] = set()
        for v in values:
            self.data[key].add(v)
        self.types[key] = 'set'
        return len(values)
    
    def smembers(self, key):
        return self.data.get(key, set())
    
    def zadd(self, key, mapping):
        if key not in self.data:
            self.data[key] = {}
        self.data[key].update(mapping)
        self.types[key] = 'zset'
        return len(mapping)
    
    def zrevrange(self, key, start, end, withscores=False):
        items = sorted(self.data.get(key, {}).items(), key=lambda x: -x[1])
        if end == -1:
            end = len(items)
        return items[start:end+1]


r = FakeRedis()

# String: 缓存
r.set('user:1:name', 'Alice')
r.set('page:home:views', 0)
r.incr('page:home:views')
r.incr('page:home:views')
r.incr('page:home:views')
print("String:")
print(f"  user:1:name = {r.get('user:1:name')}")
print(f"  page:home:views = {r.get('page:home:views')}")

# List: 消息队列
r.lpush('notifications', 'msg3', 'msg2', 'msg1')
print(f"\nList (notifications): {r.lrange('notifications', 0, -1)}")

# Hash: 对象
r.hset('user:1', {'name': 'Alice', 'age': '30', 'city': 'Beijing'})
print(f"\nHash (user:1): {r.hgetall('user:1')}")

# Set: 标签
r.sadd('user:1:tags', 'python', 'ml', 'gnn')
r.sadd('user:2:tags', 'python', 'nlp', 'llm')
print(f"\nSet (user:1:tags): {r.smembers('user:1:tags')}")
print(f"Set (user:2:tags): {r.smembers('user:2:tags')}")
print(f"交集: {r.smembers('user:1:tags') & r.smembers('user:2:tags')}")

# Sorted Set: 排行榜
r.zadd('leaderboard', {'Alice': 95, 'Bob': 87, 'Carol': 92, 'Dave': 78, 'Eve': 88})
print(f"\nSorted Set (排行榜 Top 3):")
for name, score in r.zrevrange('leaderboard', 0, 2):
    print(f"  {name}: {score}")

---
## 6. Redis 缓存策略 <a id='6'></a>

### 三大缓存问题

| 问题 | 描述 | 解决方案 |
|---|---|---|
| **缓存穿透** | 查询不存在的数据，每次都打 DB | 布隆过滤器 / 缓存空值 |
| **缓存击穿** | 热点 key 过期，大量请求打 DB | 互斥锁 / 永不过期 + 异步更新 |
| **缓存雪崩** | 大量 key 同时过期 | 随机 TTL / 多级缓存 / 限流 |

### 缓存模式

| 模式 | 流程 | 适用场景 |
|---|---|---|
| **Cache-Aside** | 读: 先缓存→miss→DB→写缓存 | 最常用 |
| **Read-Through** | 缓存层自动加载 DB | 简化业务代码 |
| **Write-Through** | 写缓存同时写 DB | 强一致性 |
| **Write-Behind** | 写缓存，异步批量写 DB | 高写入性能 |

In [ ]:
# ============ Cache-Aside 模式实现 ============

class CacheAsideSystem:
    """Cache-Aside 缓存模式演示"""
    
    def __init__(self, ttl=10):
        self.cache = {}  # key → (value, expire_time)
        self.db = {}     # 模拟数据库
        self.ttl = ttl
        self.stats = {'hit': 0, 'miss': 0}
    
    def db_query(self, key):
        """模拟 DB 查询 (慢)"""
        time.sleep(0.001)  # 模拟 1ms 延迟
        return self.db.get(key)
    
    def get(self, key):
        """读: 先查缓存 → miss 查 DB → 写缓存"""
        now = time.time()
        
        # 1. 查缓存
        if key in self.cache:
            value, expire = self.cache[key]
            if now < expire:
                self.stats['hit'] += 1
                return value
            else:
                del self.cache[key]  # 过期删除
        
        # 2. 缓存 miss → 查 DB
        self.stats['miss'] += 1
        value = self.db_query(key)
        
        # 3. 写缓存
        if value is not None:
            self.cache[key] = (value, now + self.ttl)
        
        return value
    
    def put(self, key, value):
        """写: 更新 DB → 删除缓存 (不是更新!)"""
        self.db[key] = value
        if key in self.cache:
            del self.cache[key]  # 删除缓存，而非更新


# 模拟
system = CacheAsideSystem(ttl=5)

# 预置数据
for i in range(100):
    system.db[f'user:{i}'] = f'User_{i}'

# 模拟访问 (热点: 80% 请求访问 20% 的数据)
np.random.seed(42)
hot_keys = [f'user:{i}' for i in range(20)]  # 热点
cold_keys = [f'user:{i}' for i in range(20, 100)]

for _ in range(1000):
    if np.random.rand() < 0.8:
        key = np.random.choice(hot_keys)
    else:
        key = np.random.choice(cold_keys)
    system.get(key)

total = system.stats['hit'] + system.stats['miss']
print(f"Cache-Aside 缓存效果:")
print(f"  总请求: {total}")
print(f"  缓存命中: {system.stats['hit']} ({system.stats['hit']/total:.1%})")
print(f"  缓存未命中: {system.stats['miss']} ({system.stats['miss']/total:.1%})")
print(f"  缓存大小: {len(system.cache)} keys")

# 可视化
fig, ax = plt.subplots(1, 1, figsize=(6, 4))
ax.bar(['Hit', 'Miss'], [system.stats['hit'], system.stats['miss']],
       color=[SAGE, TERRACOTTA], alpha=0.85, edgecolor='white')
ax.set_ylabel('Count')
ax.set_title(f'Cache Hit Rate: {system.stats["hit"]/total:.1%}')
plt.tight_layout()
plt.show()

---
## 7. Kafka 核心概念 <a id='7'></a>

### 架构

```
Producer → [Broker Cluster] → Consumer Group
              ↓
         Topic: orders
         ├── Partition 0: [msg0, msg3, msg6, ...]
         ├── Partition 1: [msg1, msg4, msg7, ...]
         └── Partition 2: [msg2, msg5, msg8, ...]
```

### 核心概念

| 概念 | 说明 |
|---|---|
| **Topic** | 消息主题 (逻辑分类) |
| **Partition** | Topic 的物理分区 (并行单位) |
| **Replica** | 分区副本 (高可用) |
| **Broker** | Kafka 服务器节点 |
| **Producer** | 消息生产者 (写入 Topic) |
| **Consumer** | 消息消费者 (从 Topic 读取) |
| **Consumer Group** | 消费者组 (组内负载均衡) |
| **Offset** | 消费位置 (每个分区独立维护) |

### 消息语义

| 语义 | 含义 | 实现 |
|---|---|---|
| **At-most-once** | 最多一次 (可能丢消息) | 先提交 offset 再处理 |
| **At-least-once** | 至少一次 (可能重复) | 先处理再提交 offset |
| **Exactly-once** | 精确一次 (理想) | 幂等生产者 + 事务 |

In [ ]:
# ============ Kafka 概念模拟 ============

class SimulatedKafka:
    """模拟 Kafka 核心机制"""
    
    def __init__(self, n_partitions=3):
        self.topics = {}
        self.n_partitions = n_partitions
        self.consumer_groups = {}
    
    def create_topic(self, name):
        self.topics[name] = [[] for _ in range(self.n_partitions)]
        print(f"Topic '{name}' created with {self.n_partitions} partitions")
    
    def produce(self, topic, key, value):
        """根据 key 的 hash 分配 partition"""
        partition = hash(key) % self.n_partitions
        offset = len(self.topics[topic][partition])
        self.topics[topic][partition].append({'offset': offset, 'key': key, 'value': value})
        return partition, offset
    
    def create_consumer_group(self, group_id, topic, n_consumers):
        """创建消费者组，分配 partition"""
        partitions = list(range(self.n_partitions))
        assignment = {}
        for i in range(n_consumers):
            assigned = [p for p in partitions if p % n_consumers == i]
            assignment[f'consumer_{i}'] = assigned
        self.consumer_groups[group_id] = {
            'assignment': assignment,
            'offsets': {p: 0 for p in partitions},
        }
        return assignment
    
    def consume(self, group_id, consumer_id):
        """消费消息"""
        group = self.consumer_groups[group_id]
        partitions = group['assignment'][consumer_id]
        messages = []
        for p in partitions:
            offset = group['offsets'][p]
            topic_name = list(self.topics.keys())[0]  # 简化
            partition_data = self.topics[topic_name][p]
            while offset < len(partition_data):
                messages.append(partition_data[offset])
                offset += 1
            group['offsets'][p] = offset
        return messages


# 演示
kafka = SimulatedKafka(n_partitions=3)
kafka.create_topic('orders')

# 生产消息
print("\n生产消息:")
orders = [('order_1','{"item":"phone"}'), ('order_2','{"item":"laptop"}'),
          ('order_3','{"item":"tablet"}'), ('order_4','{"item":"watch"}'),
          ('order_5','{"item":"headphone"}'), ('order_6','{"item":"camera"}')]
for key, value in orders:
    p, o = kafka.produce('orders', key, value)
    print(f"  {key} → Partition {p}, Offset {o}")

# 消费者组
print("\n消费者组 (2 个消费者):")
assignment = kafka.create_consumer_group('group_1', 'orders', 2)
for cid, parts in assignment.items():
    print(f"  {cid}: Partition {parts}")

# 消费
print("\n消费结果:")
for cid in ['consumer_0', 'consumer_1']:
    msgs = kafka.consume('group_1', cid)
    print(f"  {cid}: {[m['key'] for m in msgs]}")

print("\n💡 Kafka 核心: Partition 是并行单位，Consumer Group 实现负载均衡")
print("💡 同一 Partition 内消息有序，跨 Partition 无序")

In [ ]:
# ============ Kafka vs Redis Pub/Sub vs RabbitMQ ============

comparison = {
    '维度': ['消息模型', '持久化', '吞吐量', '延迟', '消息顺序', '消费模式', '适用场景'],
    'Kafka': ['发布订阅+队列', '持久化(磁盘)', '百万级/s', 'ms级', '分区内有序', 'Pull(消费者拉取)', '日志/事件流/大数据'],
    'Redis Pub/Sub': ['发布订阅', '无持久化', '十万级/s', 'μs级', '无保证', 'Push(推送)', '实时通知/缓存失效'],
    'RabbitMQ': ['队列', '持久化', '万级/s', 'μs级', '队列内有序', 'Push(推送)', '任务队列/RPC'],
}

print(f"{'维度':<12} {'Kafka':<22} {'Redis Pub/Sub':<22} {'RabbitMQ':<22}")
print("-" * 78)
for i, dim in enumerate(comparison['维度']):
    print(f"{dim:<12} {comparison['Kafka'][i]:<22} {comparison['Redis Pub/Sub'][i]:<22} {comparison['RabbitMQ'][i]:<22}")

In [ ]:
# ============ Mini 事件驱动管线 ============

class EventPipeline:
    """模拟事件驱动架构: Producer → Queue → Consumer → Cache"""
    
    def __init__(self):
        self.queue = deque()
        self.cache = {}
        self.db = {}
        self.processed = 0
    
    def produce(self, event_type, data):
        event = {'type': event_type, 'data': data, 'ts': time.time()}
        self.queue.append(event)
    
    def consume(self):
        if not self.queue:
            return None
        event = self.queue.popleft()
        
        # 处理事件
        if event['type'] == 'user_update':
            user_id = event['data']['user_id']
            self.db[user_id] = event['data']
            self.cache[user_id] = event['data']  # 写缓存
        elif event['type'] == 'order_created':
            order_id = event['data']['order_id']
            self.db[order_id] = event['data']
        
        self.processed += 1
        return event


pipeline = EventPipeline()

# 模拟事件流
np.random.seed(42)
for i in range(50):
    if np.random.rand() < 0.6:
        pipeline.produce('user_update', {'user_id': f'u{np.random.randint(1,10)}', 'name': f'User_{i}'})
    else:
        pipeline.produce('order_created', {'order_id': f'o{i}', 'amount': np.random.randint(10, 1000)})

# 消费所有事件
while pipeline.queue:
    pipeline.consume()

print(f"事件驱动管线模拟:")
print(f"  处理事件: {pipeline.processed}")
print(f"  DB 记录: {len(pipeline.db)}")
print(f"  缓存条目: {len(pipeline.cache)}")
print(f"  队列剩余: {len(pipeline.queue)}")
print(f"\n💡 事件驱动: 生产者和消费者解耦，各自独立扩展")

---
## 9. 总结 & 思考题 <a id='9'></a>

### 今日核心知识点

1. **SQL JOIN**: INNER/LEFT/RIGHT/FULL/CROSS，理解 NULL 的处理
2. **窗口函数**: ROW_NUMBER/RANK/LAG/SUM OVER，保持原始行数
3. **索引**: B-tree 原理，何时建/不建索引，EXPLAIN 解读
4. **Redis**: 6 种数据结构 + 4 种缓存模式 + 3 大缓存问题
5. **Kafka**: Topic/Partition/Consumer Group + 3 种消息语义
6. **选型**: Kafka(事件流) vs Redis(缓存/实时) vs RabbitMQ(任务队列)

### 面试高频问题

1. **INNER JOIN 和 LEFT JOIN 区别？** INNER 只保留匹配行，LEFT 保留左表所有行
2. **ROW_NUMBER 和 RANK 区别？** ROW_NUMBER 不重复，RANK 有并列会跳号
3. **Redis 和 Memcached 区别？** Redis 支持多种数据结构、持久化、集群
4. **缓存穿透怎么解决？** 布隆过滤器 / 缓存空值
5. **Kafka 怎么保证消息不丢？** producer acks=all + min.insync.replicas + consumer 手动提交
6. **什么时候用 Kafka 而不是 Redis？** 需要持久化、高吞吐、事件溯源时用 Kafka

In [ ]:
print("=" * 60)
print("W4 Day3 完成!")
print("=" * 60)
print("""
今日成果:
  ✅ SQL JOIN 全类型演示 (sqlite3)
  ✅ 窗口函数: ROW_NUMBER/RANK/LAG/聚合
  ✅ 索引性能实验 (10万行, EXPLAIN)
  ✅ 5 道 LeetCode SQL 解答
  ✅ Redis 6 种数据结构演示 (模拟器)
  ✅ Cache-Aside 模式实现 + 命中率统计
  ✅ Kafka 概念模拟 (Topic/Partition/Consumer Group)
  ✅ 事件驱动管线模拟

关键结论:
  • 窗口函数是 SQL 面试的高频考点，必须熟练
  • Redis 缓存三大问题 (穿透/击穿/雪崩) 是系统设计必问
  • Kafka Partition = 并行单位，Consumer Group = 负载均衡
  • AI 工程师也需要扎实的数据基础设施知识
""")